# Imports

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import time
import pandas as pd
from pathlib import Path
import yaml
from sklearn.metrics import classification_report

In [ ]:
# Load the shared configuration from config.yaml at the repository root
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'config.yaml').exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('config.yaml not found. Run this notebook from inside the repository.')
with open(REPO_ROOT / 'config.yaml') as f:
    CONFIG = yaml.safe_load(f)
PATHS = {name: REPO_ROOT / path for name, path in CONFIG['paths'].items()}

In [ ]:
if CONFIG['device'] != 'auto':
    device = torch.device(CONFIG['device'])
    print("Using device:", device)
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using GPU:", torch.cuda.get_device_name(0))
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple Silicon GPU")

else:
    device = torch.device("cpu")
    print("Using CPU")

# Configurations

In [ ]:
TASK = CONFIG['task']  # 'Suturing', 'Needle_Passing' or 'Knot_Tying'

# Configuration
NUM_SURGEONS = CONFIG['model']['num_surgeons']

#Gesture generation
GESTURE_VOCAB_SIZE = CONFIG['model']['gesture_vocab_size']  # 15 gestures + 1 MASK

SEQ_LEN = CONFIG['data']['seq_len']
HIDDEN_DIM = CONFIG['model']['hidden_dim']
VISION_DIM = CONFIG['model']['vision_dim']
LANG_DIM = CONFIG['model']['lang_dim']
NUM_TIMESTEPS = CONFIG['model']['num_timesteps']
EPOCHS = CONFIG['training']['epochs']
BATCH_SIZE = CONFIG['training']['batch_size']
LR = CONFIG['training']['lr']
SEED = CONFIG['training']['seed']

In [ ]:
TRAIN_PATH = PATHS['features_dir'] / f"jigsaws_train_{TASK}.npz"
TEST_PATH = PATHS['features_dir'] / f"jigsaws_test_{TASK}.npz"
MODEL_SAVE_PATH = PATHS['models_dir'] / f"gesture_model_{TASK}.pt"
CSV_SAVE_PATH = PATHS['training_records_dir'] / f"gesture_training_record_{TASK}.csv"
RESULTS_SAVE_PATH = PATHS['results_dir'] / f"gesture_results_{TASK}.csv"

os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)
os.makedirs(os.path.dirname(CSV_SAVE_PATH), exist_ok=True)
os.makedirs(os.path.dirname(RESULTS_SAVE_PATH), exist_ok=True)

In [ ]:
results = []
training_epochs_record = pd.DataFrame(columns=['epoch', 'loss', 'top1_acc', 'top5_acc', 'weighted_f1', 'train_time'])

## Seed

In [ ]:
# Set random seeds for reproducibility
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)
random.seed(SEED)
np.random.seed(SEED)
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

# Main

In [ ]:
# Forward corruption: transition matrices
q_matrix = np.zeros((NUM_TIMESTEPS, GESTURE_VOCAB_SIZE, GESTURE_VOCAB_SIZE))
for t in range(NUM_TIMESTEPS):
    alpha = 1 - (t + 1) / NUM_TIMESTEPS
    for i in range(GESTURE_VOCAB_SIZE):
        q_matrix[t, i] = (1 - alpha) / (GESTURE_VOCAB_SIZE - 1)
        q_matrix[t, i, i] = alpha
q_matrix = torch.tensor(q_matrix, dtype=torch.float32)

def corrupt(x0, t):
    B, T = x0.shape
    probs = q_matrix[t][:, x0.view(-1)]  # (V, B*T)
    probs = probs.permute(1, 0).view(B, T, GESTURE_VOCAB_SIZE)
    xt = torch.multinomial(probs.view(-1, GESTURE_VOCAB_SIZE), 1).view(B, T)
    return xt

class RealJIGSAWSDataset(Dataset):
    def __init__(self, npz_path):
        data = np.load(npz_path)
        self.vision_feat = torch.tensor(data["vision_feat"], dtype=torch.float32)
        self.lang_feat = torch.tensor(data["lang_feat"], dtype=torch.float32)
        self.gesture_seq = torch.tensor(data["gesture_seq"], dtype=torch.long)
        self.surgeon_id = torch.tensor(data["surgeon_id"], dtype=torch.long)
        self.timesteps = torch.randint(0, NUM_TIMESTEPS, (len(self.surgeon_id),))

    def __len__(self):
        return len(self.surgeon_id)

    def __getitem__(self, idx):
        x0 = self.gesture_seq[idx]
        t = self.timesteps[idx]
        xt = corrupt(x0.unsqueeze(0), t).squeeze(0)
        return {
            "vision_feat": self.vision_feat[idx],
            "lang_feat": self.lang_feat[idx],
            "surgeon_id": self.surgeon_id[idx],
            "x0": x0,
            "xt": xt,
            "t": t
        }

class TimeEmbedding(nn.Module):
    def __init__(self, num_timesteps, hidden_dim):
        super().__init__()
        self.embed = nn.Embedding(num_timesteps, hidden_dim)

    def forward(self, t):  # (B,)
        return self.embed(t)

class SurgeonEmbedding(nn.Module):
    def __init__(self, num_surgeons, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(num_surgeons, hidden_dim)

    def forward(self, surgeon_id):
        return self.embedding(surgeon_id)

class VLAEncoder(nn.Module):
    def __init__(self, vision_dim, lang_dim, hidden_dim):
        super().__init__()
        self.vision_proj = nn.Linear(vision_dim, hidden_dim)
        self.lang_proj = nn.Linear(lang_dim, hidden_dim)
        self.fuse = nn.Linear(2 * hidden_dim, hidden_dim)

    def forward(self, vision_feat, lang_feat):
        v = self.vision_proj(vision_feat)
        l = self.lang_proj(lang_feat)
        return self.fuse(torch.cat([v, l], dim=-1))

class DiffusionGestureModel(nn.Module):
    def __init__(self, vocab_size, hidden_dim):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, hidden_dim)
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, 2*hidden_dim),
            nn.ReLU(),
            nn.Linear(2*hidden_dim, 1*hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, vocab_size)

        )

    def forward(self, xt, cond):
        x = self.token_embed(xt)
        cond = cond.unsqueeze(1).expand_as(x)
        return self.net(x + cond)

class FullModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.vla = VLAEncoder(VISION_DIM, LANG_DIM, HIDDEN_DIM)
        self.surgeon_embed = SurgeonEmbedding(NUM_SURGEONS, HIDDEN_DIM)
        self.time_embed = TimeEmbedding(NUM_TIMESTEPS, HIDDEN_DIM)
        self.denoise = DiffusionGestureModel(GESTURE_VOCAB_SIZE, HIDDEN_DIM)

    def forward(self, xt, vision_feat, lang_feat, surgeon_id, t):
        cond = self.vla(vision_feat, lang_feat) + self.surgeon_embed(surgeon_id) + self.time_embed(t)
        return self.denoise(xt, cond)


In [ ]:
def train():
    
    train_set = RealJIGSAWSDataset(TRAIN_PATH)
    test_set = RealJIGSAWSDataset(TEST_PATH)

    print(f"Train set size: {len(train_set)}, Test set size: {len(test_set)}")

    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

    model = FullModel()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()

    # send model to device
    model.to(device)

    # time calculation variable
    train_time = 0.0
    if device.type == 'cuda':
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0.0
        train_acc = 0.0
        train_top5 = 0.0
        all_preds = []
        all_labels = []

        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False):
            
            # extract batch data
            vision_feat = batch["vision_feat"]
            lang_feat = batch["lang_feat"]
            surgeon_id = batch["surgeon_id"]
            x0 = batch["x0"]
            xt = batch["xt"]
            t = batch["t"]

            # send data to device
            vision_feat = vision_feat.to(device)
            lang_feat = lang_feat.to(device)
            surgeon_id = surgeon_id.to(device)
            x0 = x0.to(device)
            xt = xt.to(device)
            t = t.to(device)

            # measure time for training step
            if device.type == 'cuda':
                start.record()
            else:
                train_start_time = time.time()

            # actual training step
            logits = model(xt, vision_feat, lang_feat, surgeon_id, t)
            loss = criterion(logits.view(-1, GESTURE_VOCAB_SIZE), x0.view(-1))

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # record end times
            if device.type == 'cuda':
                end.record()
                torch.cuda.synchronize()
                train_time += start.elapsed_time(end) / 1000.0  # convert ms to s
            else:
                train_end_time = time.time()
                train_time += (train_end_time - train_start_time)

            with torch.no_grad():
                preds = torch.argmax(logits, dim=-1)  # Top-1
                correct = (preds == x0).float().mean().item()
                train_acc += correct

                top5_preds = torch.topk(logits, k=5, dim=-1).indices
                match_top5 = top5_preds.eq(x0.unsqueeze(-1)).any(dim=-1).float().mean().item()
                train_top5 += match_top5

                all_preds.extend(preds.view(-1).cpu().numpy())
                all_labels.extend(x0.view(-1).cpu().numpy())

            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)
        avg_train_acc = train_acc / len(train_loader)
        avg_train_top5 = train_top5 / len(train_loader)
        weighted_f1 = classification_report(all_labels, all_preds, labels=list(range(GESTURE_VOCAB_SIZE)), output_dict=True, zero_division=0)['weighted avg']['f1-score']

        print(f"Epoch {epoch+1} Loss: {avg_train_loss:.4f} | Top-1 Acc: {avg_train_acc:.4f} | Top-5 Acc: {avg_train_top5:.4f}  | Weighted F1: {weighted_f1:.4f} | Train Time: {train_time:.2f}s")
        # Record training progress
        training_epochs_record.loc[len(training_epochs_record)] = [epoch+1, avg_train_loss, avg_train_acc, avg_train_top5, weighted_f1, train_time]



    # Testing loop
    model.eval()
    with torch.no_grad():
        test_loss = 0.0
        test_acc = 0.0
        test_top5 = 0.0
        all_preds = []
        all_labels = []
        for batch in tqdm(test_loader, desc="Testing", leave=False):

            # extract batch data
            vision_feat = batch["vision_feat"]
            lang_feat = batch["lang_feat"]
            surgeon_id = batch["surgeon_id"]
            x0 = batch["x0"]
            xt = batch["xt"]
            t = batch["t"]

            # send data to device
            vision_feat = vision_feat.to(device)
            lang_feat = lang_feat.to(device)
            surgeon_id = surgeon_id.to(device)
            x0 = x0.to(device)
            xt = xt.to(device)
            t = t.to(device)

            logits = None

            B, L = xt.shape

            # iterative reverse diffusion 
            for step in range(NUM_TIMESTEPS):
                time_step = NUM_TIMESTEPS - 1 - step
                t_step_tensor = torch.full((x0.size(0),), time_step, dtype=torch.long, device=device)
                logits = model(xt, vision_feat, lang_feat, surgeon_id, t_step_tensor)
                probs = torch.softmax(logits, dim=-1)
                xt = torch.multinomial(probs.view(-1, GESTURE_VOCAB_SIZE), 1).view(B, L)
            
            loss = criterion(logits.view(-1, GESTURE_VOCAB_SIZE), x0.view(-1))
            preds = torch.argmax(logits, dim=-1)  # Top-1
            correct = (preds == x0).float().mean().item()
            test_acc += correct

            top5_preds = torch.topk(logits, k=5, dim=-1).indices
            match_top5 = top5_preds.eq(x0.unsqueeze(-1)).any(dim=-1).float().mean().item()
            test_top5 += match_top5

            all_preds.extend(preds.view(-1).cpu().numpy())
            all_labels.extend(x0.view(-1).cpu().numpy())

            test_loss += loss.item()

        avg_test_loss = test_loss / len(test_loader)
        avg_test_acc = test_acc / len(test_loader)
        avg_test_top5 = test_top5 / len(test_loader)
        test_weighted_f1 = classification_report(all_labels, all_preds, labels=list(range(GESTURE_VOCAB_SIZE)), output_dict=True, zero_division=0)['weighted avg']['f1-score']

        print(f"\nTest Loss: {avg_test_loss:.4f} | Top-1 Acc: {avg_test_acc:.4f} | Top-5 Acc: {avg_test_top5:.4f}  | Weighted F1: {test_weighted_f1:.4f}")

        # Optional: print classification report
        print(classification_report(all_labels, all_preds, labels=list(range(GESTURE_VOCAB_SIZE)), zero_division=0))

    # Save results
    results.append({
        'kind': 'Gesture',
        'task': TASK,
        'type': 'from scratch',
        'accuracy': avg_test_acc,
        'top5_accuracy': avg_test_top5,
        'weighted_f1': test_weighted_f1,
        'train_time': train_time
    })

    return model, train_time


## Training

In [ ]:
model, train_time = train()
torch.save(model.state_dict(), MODEL_SAVE_PATH)

In [ ]:
print(f"train time: {train_time}")

# Save test results (read by notebooks/utils/pareto_front.ipynb and plots.ipynb)
pd.DataFrame(results).to_csv(RESULTS_SAVE_PATH, index=False)

training_epochs_record.to_csv(CSV_SAVE_PATH, index=False)